# Nova App Analytics

## Local Demand Drivers: Regression Modeling

This notebook builds a first demand modeling workflow for Nova's geography analysis.

The goal is to understand how local market context, calendar patterns, service supply, and weather relate to daily demand across Nova markets and service categories.

We will keep the workflow intentionally simple and readable:

1. Define the business problem and modeling task.
2. Load the market-category-day feature mart from BigQuery.
3. Run basic data checks.
4. Split the data chronologically.
5. Compare a baseline, Ridge regression, and Random Forest regression.
6. Evaluate predictions with clear metrics and residuals.
7. Interpret the results in business language.

Clustering is intentionally left out of this notebook. It should be handled later in a separate notebook backed by a market-grain dbt mart.

## 1. Business Problem and ML Framing

The dashboard question for this part of the project is:

> Which local conditions help explain daily Nova demand across markets and categories?

For this first notebook, we model **daily transactions**.

- **Business target:** daily demand volume.
- **ML target column:** `transactions`.
- **Problem type:** supervised regression.
- **Row grain:** one row per `market_id + category + order_date`.
- **Validation approach:** train on earlier dates and validate on later dates.

This is not a production forecasting system. It is an analysis notebook that helps us learn which features are useful, where the model performs well, and where daily demand is harder to explain.

## 2. Model Evaluation Concepts

A regression model predicts a continuous number. Here, the number is daily transactions.

The basic comparison is:

```text
error = actual transactions - predicted transactions
```

This error is also called a **residual**.

- Positive residual: the model underpredicted demand.
- Negative residual: the model overpredicted demand.
- Residual near zero: the model was close.

We evaluate on validation data because training performance can be misleading. A model can memorize the training period but fail on later dates. Since this is time-based data, we use a chronological split instead of a random split.

### Metrics Used

We will compare models using four metrics.

- **MAE:** average absolute error in transaction units. If MAE is 25, predictions are off by 25 transactions on average.
- **RMSE:** similar to MAE, but it punishes large misses more strongly.
- **WMAPE:** total absolute error divided by total actual demand. This is useful when markets/categories have different demand scales.
- **R2:** how much better the model is than a simple average-style guess. It is useful context, but it is not the main decision rule here.

The baseline model matters. A machine learning model is only useful if it beats a simple benchmark.

## 3. Setup

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
import pandas_gbq
import plotly.express as px
import plotly.io as pio

from google.cloud import bigquery
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

px.defaults.template = "plotly_white"
px.defaults.width = 1_050
px.defaults.height = 560

# If charts do not render in your environment, try:
# pio.renderers.default = "browser"
pio.renderers.default = "notebook_connected"

## 4. Load the Demand Feature Mart

The source table is the dbt mart built for this analysis:

`mart_geo_market_category_day_features`

Expected grain:

```text
market_id + category + order_date
```

If authentication fails, run this outside the notebook and retry:

```bash
gcloud auth application-default login
```

In [2]:
PROJECT_ID = "nova-project-498911"
LOCATION = "EU"
DATASET_ID = "dbt_doruk"
TABLE_NAME = "mart_geo_market_category_day_features"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)


def table_ref(table_name: str) -> str:
    return f"`{PROJECT_ID}.{DATASET_ID}.{table_name}`"


def read_bq(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()


sql = f"""
select *
from {table_ref(TABLE_NAME)}
order by order_date, market_id, category
"""

df = read_bq(sql)
df["order_date"] = pd.to_datetime(df["order_date"])
df = df.sort_values(["order_date", "market_id", "category"]).reset_index(drop=True)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")

df.head()

/Users/doruk/dev/nova-analytics/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Rows: 29,280
Columns: 80
Date range: 2024-01-01 to 2024-12-31


,market_id,market_name,nova_region,latitude,longitude,market_weight,growth_multiplier,configured_ios_share,country_name,country_iso2,country_iso3,country_region,country_subregion,currency_code,population_total,gdp_per_capita_current_usd,urban_population_pct,internet_users_pct,mobile_subscriptions_per_100_people,category,order_date,year,quarter,month,year_month,day_of_week,day_name,is_weekend,transactions,gmv_usd,avg_transaction_amount_usd,active_users,active_services_with_transactions,completed_transactions,failed_transactions,refunded_transactions,completion_rate,failed_rate,refunded_rate,promo_transactions,promo_share,head_tier_transactions,mid_tier_transactions,long_tail_tier_transactions,head_tier_transaction_share,mid_tier_transaction_share,long_tail_tier_transaction_share,ios_transactions,android_transactions,lite_transactions,web_transactions,ios_transaction_share,android_transaction_share,lite_transaction_share,web_transaction_share,active_services,avg_service_rating,avg_popularity_score,head_services,mid_services,long_tail_services,head_service_share,mid_service_share,long_tail_service_share,weather_code,temperature_2m_mean_c,temperature_2m_max_c,temperature_2m_min_c,apparent_temperature_mean_c,precipitation_sum_mm,rain_sum_mm,precipitation_hours,wind_speed_10m_max_kmh,is_rain_day,temperature_band,precipitation_band,avg_market_category_daily_transactions,p75_market_category_daily_transactions,is_high_demand_day,demand_index_vs_market_category_avg
0,1,Singapore,Southeast Asia,1.3521,103.8198,0.0700,1.0800,0.5500,Singapore,SG,SGP,Asia,South-Eastern Asia,SGD,"6,036,860.0000","90,674.0666",100.0000,94.3776,170.7826,Digital Wallet,2024-01-01,2024,1,1,2024-01,2,Monday,False,1051,"64,524.5300",61.3935,1010,255,1004,41,6,0.9553,0.0390,0.0057,0,0.0000,489,215,347,0.4653,0.2046,0.3302,436,279,121,215,0.4148,0.2655,0.1151,0.2046,418,4.3749,3.2619,11,55,352,0.0263,0.1316,0.8421,63,25.5000,29.0000,23.1000,30.5000,12.5000,12.5000,13.0000,14.7000,True,warm,moderate,"1,065.0683","1,143.0000",False,0.9868
1,1,Singapore,Southeast Asia,1.3521,103.8198,0.0700,1.0800,0.5500,Singapore,SG,SGP,Asia,South-Eastern Asia,SGD,"6,036,860.0000","90,674.0666",100.0000,94.3776,170.7826,E-Commerce,2024-01-01,2024,1,1,2024-01,2,Monday,False,1512,"171,272.6500",113.2756,1487,493,1339,62,111,0.8856,0.0410,0.0734,0,0.0000,490,485,537,0.3241,0.3208,0.3552,636,404,186,286,0.4206,0.2672,0.1230,0.1892,811,4.3998,2.9550,18,153,640,0.0222,0.1887,0.7891,63,25.5000,29.0000,23.1000,30.5000,12.5000,12.5000,13.0000,14.7000,True,warm,moderate,"2,226.8989","2,412.2500",False,0.6790
2,1,Singapore,Southeast Asia,1.3521,103.8198,0.0700,1.0800,0.5500,Singapore,SG,SGP,Asia,South-Eastern Asia,SGD,"6,036,860.0000","90,674.0666",100.0000,94.3776,170.7826,Food Delivery,2024-01-01,2024,1,1,2024-01,2,Monday,False,1921,"65,855.6400",34.2820,1837,666,1816,70,35,0.9453,0.0364,0.0182,0,0.0000,509,613,799,0.2650,0.3191,0.4159,783,557,255,326,0.4076,0.2900,0.1327,0.1697,1062,4.3758,2.4929,17,189,856,0.0160,0.1780,0.8060,63,25.5000,29.0000,23.1000,30.5000,12.5000,12.5000,13.0000,14.7000,True,warm,moderate,"2,687.1202","2,933.5000",False,0.7149
3,1,Singapore,Southeast Asia,1.3521,103.8198,0.0700,1.0800,0.5500,Singapore,SG,SGP,Asia,South-Eastern Asia,SGD,"6,036,860.0000","90,674.0666",100.0000,94.3776,170.7826,Grocery,2024-01-01,2024,1,1,2024-01,2,Monday,False,1072,"94,478.7100",88.1331,1012,347,956,67,49,0.8918,0.0625,0.0457,0,0.0000,351,287,434,0.3274,0.2677,0.4049,445,272,131,224,0.4151,0.2537,0.1222,0.2090,577,4.3887,2.6117,12,95,470,0.0208,0.1646,0.8146,63,25.5000,29.0000,23.1000,30.5000,12.5000,12.5000,13.0000,14.7000,True,warm,moderate,"1,450.3880","1,615.5000",False,0.7391
4,1,Singapore,Southeast Asia,1.3521,103.8198,0.0700,1.0800,0.5500,Singapore,SG,SGP,Asia,South-Eastern Asia,SGD,"6,036,860.0000","90,674.0666",100.0000,94.3776,170.7826,Ride Hailing,2024-01-01,2024,1,1,2024-01,2,Monday,False,1180,"38,634.7800",32.7413,1161,347,1077,79,24,0.9127,0.0669,0.0203,0,0.0000,544,284,352

## 5. Basic Checks

This is not a full EDA section. The goal is to confirm the table is usable for modeling.

We check:

- Required columns exist.
- The expected grain is unique.
- The target has sensible values.
- There are no surprising missing values in likely model features.

In [3]:
required_columns = ["market_id", "market_name", "category", "order_date", "transactions"]
missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

grain_cols = ["market_id", "category", "order_date"]
duplicate_grain_rows = int(df.duplicated(grain_cols).sum())

print(f"Duplicate grain rows: {duplicate_grain_rows:,}")
if duplicate_grain_rows > 0:
    raise ValueError("The modeling grain is not unique.")

coverage = pd.DataFrame(
    {
        "metric": ["markets", "categories", "dates", "rows"],
        "value": [
            df["market_id"].nunique(),
            df["category"].nunique(),
            df["order_date"].nunique(),
            len(df),
        ],
    }
)

coverage

Duplicate grain rows: 0


,metric,value
0,markets,16
1,categories,5
2,dates,366
3,rows,29280


In [4]:
target_summary = df["transactions"].describe().to_frame("transactions")
display(target_summary)

if (df["transactions"] < 0).any():
    raise ValueError("Transactions contains negative values.")

missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_values")
    .query("missing_values > 0")
    .sort_values("missing_values", ascending=False)
)

missing_summary

,transactions
count,"29,280.0000"
mean,"1,707.6503"
std,840.1855
min,214.0000
25%,"1,101.0000"
50%,"1,565.0000"
75%,"2,124.0000"
max,"6,653.0000"


,missing_values


## 6. Demand Shape Before Modeling

Before fitting models, we should look at the target itself.

These charts answer basic questions:

- How does total demand move over time?
- Which categories have larger or smaller daily demand?
- Are market/category scales very different?

In [5]:
daily_total = (
    df.groupby("order_date", as_index=False)
    .agg(transactions=("transactions", "sum"))
)

fig = px.line(
    daily_total,
    x="order_date",
    y="transactions",
    title="Total Daily Transactions Across Nova Markets",
    labels={"order_date": "Date", "transactions": "Transactions"},
)
fig.update_layout(hovermode="x unified")
fig.show()

In [6]:
category_summary = (
    df.groupby("category", as_index=False)
    .agg(
        total_transactions=("transactions", "sum"),
        avg_daily_transactions=("transactions", "mean"),
        median_daily_transactions=("transactions", "median"),
    )
    .sort_values("total_transactions", ascending=False)
)

display(category_summary)

fig = px.box(
    df,
    x="category",
    y="transactions",
    color="category",
    title="Daily Transaction Distribution by Category",
    labels={"category": "Category", "transactions": "Daily transactions"},
)
fig.update_layout(showlegend=False)
fig.show()

,category,total_transactions,avg_daily_transactions,median_daily_transactions
2,Food Delivery,14864102,"2,538.2688","2,444.0000"
4,Ride Hailing,11202731,"1,913.0347","1,857.0000"
1,E-Commerce,11005827,"1,879.4103","1,768.0000"
3,Grocery,6915310,"1,180.8931","1,163.0000"
0,Digital Wallet,6012030,"1,026.6445",920.0000


## 7. Feature Framing

For the first model, we use **prediction-safe** features: values that are known before or at the start of the day, or relatively static context.

Examples:

- Market and category identifiers.
- Calendar fields.
- Market context and macro indicators.
- Service supply fields.
- Weather fields.

We intentionally exclude same-day operational outcomes and target-derived fields from the main model.

Examples we do **not** use in v1:

- `active_users`
- status counts and rates like `completion_rate`, `failed_rate`, `completed_transactions`
- platform transaction shares like `ios_transaction_share`
- target-derived fields like `avg_market_category_daily_transactions`, `p75_market_category_daily_transactions`, `is_high_demand_day`, and `demand_index_vs_market_category_avg`

Those fields can be useful for explanation, but they are not clean predictors for a forward-looking demand model.

In [7]:
target_col = "transactions"

metadata_cols = [
    "market_id",
    "market_name",
    "nova_region",
    "country_name",
    "country_iso3",
    "category",
    "order_date",
]

categorical_features = [
    "market_id",
    "nova_region",
    "country_iso3",
    "category",
    "day_name",
    "weather_code",
    "temperature_band",
    "precipitation_band",
]

numeric_features = [
    "latitude",
    "longitude",
    "market_weight",
    "growth_multiplier",
    "configured_ios_share",
    "population_total",
    "gdp_per_capita_current_usd",
    "urban_population_pct",
    "internet_users_pct",
    "mobile_subscriptions_per_100_people",
    "year",
    "quarter",
    "month",
    "day_of_week",
    "is_weekend",
    "active_services",
    "avg_service_rating",
    "avg_popularity_score",
    "head_services",
    "mid_services",
    "long_tail_services",
    "head_service_share",
    "mid_service_share",
    "long_tail_service_share",
    "temperature_2m_mean_c",
    "temperature_2m_max_c",
    "temperature_2m_min_c",
    "apparent_temperature_mean_c",
    "precipitation_sum_mm",
    "rain_sum_mm",
    "precipitation_hours",
    "wind_speed_10m_max_kmh",
    "is_rain_day",
]

available_categorical_features = [col for col in categorical_features if col in df.columns]
available_numeric_features = [col for col in numeric_features if col in df.columns]
feature_cols = available_categorical_features + available_numeric_features

model_columns = list(dict.fromkeys(metadata_cols + [target_col] + feature_cols))
model_df = df[model_columns].copy()

for col in ["weather_code"]:
    if col in model_df.columns:
        model_df[col] = model_df[col].astype("string")

for col in ["is_weekend", "is_rain_day"]:
    if col in model_df.columns:
        model_df[col] = model_df[col].astype("int64")

print(f"Categorical features: {len(available_categorical_features)}")
print(f"Numeric features: {len(available_numeric_features)}")
print(f"Total model features: {len(feature_cols)}")

pd.DataFrame({"feature": feature_cols})

Categorical features: 8
Numeric features: 33
Total model features: 41


,feature
0,market_id
1,nova_region
2,country_iso3
3,category
4,day_name
5,weather_code
6,temperature_band
7,precipitation_band
8,latitude
9,longitude


## 8. Chronological Train/Validation Split

Because this is time-based demand data, we avoid a random split.

- **Train:** January through October 2024.
- **Validation:** November through December 2024.

The model learns from earlier dates and is evaluated on later dates.

In [8]:
validation_start = pd.Timestamp("2024-11-01")

train = model_df[model_df["order_date"] < validation_start].copy()
valid = model_df[model_df["order_date"] >= validation_start].copy()

print(f"Train period:      {train['order_date'].min().date()} to {train['order_date'].max().date()}")
print(f"Validation period: {valid['order_date'].min().date()} to {valid['order_date'].max().date()}")
print(f"Train rows:        {len(train):,}")
print(f"Validation rows:   {len(valid):,}")

if train.empty or valid.empty:
    raise ValueError("Train or validation split is empty. Check the date range.")

Train period:      2024-01-01 to 2024-10-31
Validation period: 2024-11-01 to 2024-12-31
Train rows:        24,400
Validation rows:   4,880


In [9]:
split_plot = pd.concat(
    [
        train.assign(split="Train"),
        valid.assign(split="Validation"),
    ],
    ignore_index=True,
)

split_daily = (
    split_plot.groupby(["order_date", "split"], as_index=False)
    .agg(transactions=("transactions", "sum"))
)

fig = px.line(
    split_daily,
    x="order_date",
    y="transactions",
    color="split",
    title="Chronological Train/Validation Split",
    labels={"order_date": "Date", "transactions": "Transactions", "split": "Split"},
)
fig.update_layout(hovermode="x unified")
fig.show()

## 9. Define Candidate Models

We compare three models.

### Model A: Market-Category Average Baseline

This model predicts the training-period average transactions for each market/category pair.

This is the minimum bar. If a more advanced model cannot beat it, the extra complexity is not useful.

### Model B: Ridge Regression

Ridge regression is a regularized linear model. It is useful because it is relatively easy to interpret and less likely to overfit than plain linear regression.

### Model C: Random Forest Regression

Random Forest can capture nonlinear patterns and feature interactions. It is less directly interpretable than Ridge, but feature importance helps us understand which inputs matter most.

## 10. Evaluation Functions

The helper functions below keep evaluation consistent across the baseline and machine learning models.

In [10]:
def wmape(y_true: pd.Series | np.ndarray, y_pred: pd.Series | np.ndarray) -> float:
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)
    denominator = np.abs(y_true_arr).sum()
    if denominator == 0:
        return np.nan
    return np.abs(y_true_arr - y_pred_arr).sum() / denominator


def evaluate_predictions(model_name: str, y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float | str]:
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "WMAPE": wmape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def build_prediction_frame(frame: pd.DataFrame, model_name: str, y_pred: np.ndarray) -> pd.DataFrame:
    predictions = frame[metadata_cols + [target_col]].copy()
    predictions["model"] = model_name
    predictions["actual"] = predictions[target_col].astype(float)
    predictions["predicted"] = np.maximum(y_pred.astype(float), 0)
    predictions["residual"] = predictions["actual"] - predictions["predicted"]
    predictions["abs_error"] = predictions["residual"].abs()
    predictions["abs_pct_error"] = np.where(
        predictions["actual"] == 0,
        np.nan,
        predictions["abs_error"] / predictions["actual"],
    )
    return predictions.drop(columns=[target_col])

## 11. Train and Evaluate Models

The Ridge and Random Forest models are trained on `log1p(transactions)`.

This makes the model learn relative demand patterns more smoothly and avoids negative predictions after converting back with `expm1`.

In [11]:
baseline_lookup = (
    train.groupby(["market_id", "category"], as_index=False)
    .agg(baseline_prediction=(target_col, "mean"))
)
global_train_mean = train[target_col].mean()

baseline_valid = valid[["market_id", "category"]].merge(
    baseline_lookup,
    on=["market_id", "category"],
    how="left",
)
baseline_pred = baseline_valid["baseline_prediction"].fillna(global_train_mean).to_numpy()

metrics = [evaluate_predictions("market_category_average_baseline", valid[target_col], baseline_pred)]
prediction_frames = [build_prediction_frame(valid, "market_category_average_baseline", baseline_pred)]

pd.DataFrame(metrics)

,model,MAE,RMSE,WMAPE,R2
0,market_category_average_baseline,618.9023,786.9641,0.2784,0.3757


In [12]:
X_train = train[feature_cols]
X_valid = valid[feature_cols]
y_train = train[target_col].astype(float)
y_valid = valid[target_col].astype(float)
y_train_log = np.log1p(y_train)

def make_preprocessor() -> ColumnTransformer:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, available_numeric_features),
            ("categorical", categorical_transformer, available_categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

models = {
    "ridge_regression": Pipeline(
        steps=[
            ("preprocessor", make_preprocessor()),
            ("model", Ridge(alpha=1.0)),
        ]
    ),
    "random_forest_regression": Pipeline(
        steps=[
            ("preprocessor", make_preprocessor()),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    max_depth=14,
                    min_samples_leaf=8,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

fitted_models = {}

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train_log)
    pred = np.expm1(pipeline.predict(X_valid))
    pred = np.maximum(pred, 0)

    fitted_models[model_name] = pipeline
    metrics.append(evaluate_predictions(model_name, y_valid, pred))
    prediction_frames.append(build_prediction_frame(valid, model_name, pred))

metrics_df = pd.DataFrame(metrics).sort_values("WMAPE").reset_index(drop=True)
validation_predictions = pd.concat(prediction_frames, ignore_index=True)

metrics_display = metrics_df.copy()
metrics_display["WMAPE_percent"] = metrics_display["WMAPE"] * 100
metrics_display

,model,MAE,RMSE,WMAPE,R2,WMAPE_percent
0,ridge_regression,345.1609,520.5791,0.1552,0.7268,15.5243
1,random_forest_regression,384.4837,551.3278,0.1729,0.6936,17.2929
2,market_category_average_baseline,618.9023,786.9641,0.2784,0.3757,27.8363


## 12. Model Comparison

Lower MAE, RMSE, and WMAPE are better.

The baseline is especially important. If a model does not beat the market-category average, it is not adding enough value for this task.

In [13]:
fig = px.bar(
    metrics_display.sort_values("WMAPE_percent"),
    x="model",
    y="WMAPE_percent",
    color="model",
    title="Validation WMAPE by Model",
    labels={"model": "Model", "WMAPE_percent": "WMAPE (%)"},
)
fig.update_layout(showlegend=False, xaxis_tickangle=-25)
fig.show()

best_model_name = metrics_df.iloc[0]["model"]
print(f"Best validation model by WMAPE: {best_model_name}")

Best validation model by WMAPE: ridge_regression


## 13. Actual vs Predicted Demand

Metrics summarize performance, but the plots show when the model was right or wrong.

The first charts use daily validation rows as model diagnostics. For the dashboard narrative, we will create a smoother monthly aggregate in the next section.

In [14]:
selected_predictions = validation_predictions[
    validation_predictions["model"] == best_model_name
].copy()

daily_actual_predicted = (
    selected_predictions.groupby("order_date", as_index=False)
    .agg(actual=("actual", "sum"), predicted=("predicted", "sum"))
    .melt(id_vars="order_date", value_vars=["actual", "predicted"], var_name="series", value_name="transactions")
)

fig = px.line(
    daily_actual_predicted,
    x="order_date",
    y="transactions",
    color="series",
    title=f"Validation Actual vs Predicted Daily Transactions: {best_model_name}",
    labels={"order_date": "Date", "transactions": "Transactions", "series": "Series"},
)
fig.update_layout(hovermode="x unified")
fig.show()

In [15]:
fig = px.scatter(
    selected_predictions,
    x="actual",
    y="predicted",
    color="category",
    hover_data=["market_name", "order_date"],
    title=f"Validation Rows: Actual vs Predicted Transactions ({best_model_name})",
    labels={"actual": "Actual transactions", "predicted": "Predicted transactions"},
)

axis_max = max(selected_predictions["actual"].max(), selected_predictions["predicted"].max())
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=axis_max,
    y1=axis_max,
    line=dict(color="black", dash="dash"),
)
fig.show()

## 14. Monthly Dashboard Output

Daily demand is useful for model training because weather and weekday effects happen at daily resolution.

For the dashboard, daily lines can look too noisy and synthetic. The cleaner presentation layer is a monthly aggregate built from the selected model's validation predictions.

This section creates a Tableau-ready validation output with:

- monthly actual transactions
- monthly predicted transactions
- monthly residuals
- monthly WMAPE
- market and category dimensions

This is still a notebook output, not a BigQuery export.

In [16]:
monthly_dashboard_output = selected_predictions.copy()
monthly_dashboard_output["year_month"] = monthly_dashboard_output["order_date"].dt.to_period("M").astype(str)

monthly_dashboard_output = (
    monthly_dashboard_output.groupby(["year_month", "market_name", "category", "model"], as_index=False)
    .agg(
        validation_days=("order_date", "nunique"),
        actual_transactions=("actual", "sum"),
        predicted_transactions=("predicted", "sum"),
        residual_transactions=("residual", "sum"),
        absolute_error=("abs_error", "sum"),
    )
)
monthly_dashboard_output["wmape"] = np.where(
    monthly_dashboard_output["actual_transactions"] == 0,
    np.nan,
    monthly_dashboard_output["absolute_error"] / monthly_dashboard_output["actual_transactions"],
)
monthly_dashboard_output["residual_direction"] = np.select(
    [
        monthly_dashboard_output["residual_transactions"] > 0,
        monthly_dashboard_output["residual_transactions"] < 0,
    ],
    ["Underpredicted", "Overpredicted"],
    default="On target",
)

# Reconciliation check: monthly totals should match selected daily validation totals.
monthly_actual_total = monthly_dashboard_output["actual_transactions"].sum()
daily_actual_total = selected_predictions["actual"].sum()
monthly_predicted_total = monthly_dashboard_output["predicted_transactions"].sum()
daily_predicted_total = selected_predictions["predicted"].sum()

if not np.isclose(monthly_actual_total, daily_actual_total):
    raise ValueError("Monthly actual totals do not reconcile to daily validation actuals.")
if not np.isclose(monthly_predicted_total, daily_predicted_total):
    raise ValueError("Monthly predicted totals do not reconcile to daily validation predictions.")

print(f"Rows: {len(monthly_dashboard_output):,}")
print(f"Monthly actual total: {monthly_actual_total:,.0f}")
print(f"Monthly predicted total: {monthly_predicted_total:,.0f}")

monthly_dashboard_output.head(10)

Rows: 160
Monthly actual total: 10,850,000
Monthly predicted total: 9,644,370


,year_month,market_name,category,model,validation_days,actual_transactions,predicted_transactions,residual_transactions,absolute_error,wmape,residual_direction
0,2024-11,Bangalore,Digital Wallet,ridge_regression,30,"59,307.0000","44,572.4758","14,734.5242","14,734.5242",0.2484,Underpredicted
1,2024-11,Bangalore,E-Commerce,ridge_regression,30,"113,421.0000","75,600.9727","37,820.0273","37,820.0273",0.3334,Underpredicted
2,2024-11,Bangalore,Food Delivery,ridge_regression,30,"98,007.0000","91,478.9762","6,528.0238","7,379.4484",0.0753,Underpredicted
3,2024-11,Bangalore,Grocery,ridge_regression,30,"46,838.0000","48,482.6299","-1,644.6299","2,885.4267",0.0616,Overpredicted
4,2024-11,Bangalore,Ride Hailing,ridge_regression,30,"84,923.0000","88,157.6701","-3,234.6701","6,580.7375",0.0775,Overpredicted
5,2024-11,Bangkok,Digital Wallet,ridge_regression,30,"40,989.0000","35,482.2521","5,506.7479","5,506.7479",0.1343,Underpredicted
6,2024-11,Bangkok,E-Commerce,ridge_regression,30,"79,340.0000","56,948.0521","22,391.9479","22,391.9479",0.2822,Underpredicted
7,2024-11,Bangkok,Food Delivery,ridge_regression,30,"105,478.0000","100,553.4358","4,924.5642","5,701.2235",0.0541,Underpredicted
8,2024-11,Bangkok,Grocery,ridge_regression,30,"40,607.0000","38,182.5700","2,424.4300","2,688.1130",0.0662,Underpredicted
9,2024-11,Bangkok,Ride Hailing,ridge_regression,30,"67,168.0000","71,686.7304","-4,518.7304","5,087.0255",0.0757,Overpredicted


In [17]:
monthly_total = (
    monthly_dashboard_output.groupby("year_month", as_index=False)
    .agg(
        actual_transactions=("actual_transactions", "sum"),
        predicted_transactions=("predicted_transactions", "sum"),
        absolute_error=("absolute_error", "sum"),
    )
)
monthly_total["wmape"] = monthly_total["absolute_error"] / monthly_total["actual_transactions"]

monthly_total_long = monthly_total.melt(
    id_vars=["year_month", "wmape"],
    value_vars=["actual_transactions", "predicted_transactions"],
    var_name="series",
    value_name="transactions",
)

fig = px.line(
    monthly_total_long,
    x="year_month",
    y="transactions",
    color="series",
    markers=True,
    title=f"Monthly Validation Demand: Actual vs Predicted ({best_model_name})",
    labels={"year_month": "Month", "transactions": "Transactions", "series": "Series"},
)
fig.update_layout(hovermode="x unified")
fig.show()

monthly_total

,year_month,actual_transactions,predicted_transactions,absolute_error,wmape
0,2024-11,"5,200,000.0000","4,664,688.0014","820,944.3666",0.1579
1,2024-12,"5,650,000.0000","4,979,681.6483","863,440.7276",0.1528


In [18]:
monthly_category = (
    monthly_dashboard_output.groupby(["year_month", "category"], as_index=False)
    .agg(
        actual_transactions=("actual_transactions", "sum"),
        predicted_transactions=("predicted_transactions", "sum"),
        absolute_error=("absolute_error", "sum"),
    )
)
monthly_category["wmape"] = monthly_category["absolute_error"] / monthly_category["actual_transactions"]

fig = px.bar(
    monthly_category,
    x="year_month",
    y="actual_transactions",
    color="category",
    title="Monthly Actual Demand by Category",
    labels={"year_month": "Month", "actual_transactions": "Actual transactions", "category": "Category"},
)
fig.show()

monthly_market_error = (
    monthly_dashboard_output.groupby("market_name", as_index=False)
    .agg(
        actual_transactions=("actual_transactions", "sum"),
        absolute_error=("absolute_error", "sum"),
        residual_transactions=("residual_transactions", "sum"),
    )
)
monthly_market_error["wmape"] = monthly_market_error["absolute_error"] / monthly_market_error["actual_transactions"]
monthly_market_error = monthly_market_error.sort_values("wmape", ascending=False)

fig = px.bar(
    monthly_market_error.sort_values("wmape", ascending=True),
    x="wmape",
    y="market_name",
    orientation="h",
    title="Monthly Validation Error by Market",
    labels={"wmape": "WMAPE", "market_name": "Market"},
)
fig.update_layout(xaxis_tickformat=".1%")
fig.show()

monthly_market_error

,market_name,actual_transactions,absolute_error,residual_transactions,wmape
10,New York,"672,192.0000","124,015.0080","114,139.9977",0.1845
6,London,"641,349.0000","116,715.8544","107,943.6470",0.1820
14,Sydney,"251,692.0000","45,764.2615","37,341.2109",0.1818
15,Tokyo,"609,347.0000","103,552.8107","75,498.4579",0.1699
5,Jakarta,"1,009,633.0000","171,031.5174","52,168.2608",0.1694
7,Manila,"832,019.0000","140,161.0542","61,536.3181",0.1685
0,Bangalore,"821,113.0000","133,367.1763","101,135.4551",0.1624
13,Singapore,"714,102.0000","112,244.7201","51,237.2065",0.1572
2,Dubai,"541,419.0000","82,716.3576","71,021.2240",0.1528
9,Mumbai,"941,813.0000","139,810.3627","112,714.3722",0.1484


## 15. Residual Diagnostics

Residuals are calculated as:

```text
actual - predicted
```

The goal is not to get every residual to zero. The goal is to understand whether errors are systematic.

Questions to ask:

- Does the model underpredict certain dates?
- Does it struggle with specific markets?
- Does it struggle with specific categories?

The dashboard should use monthly residuals where possible, while these detailed validation diagnostics help us understand model behavior.

In [19]:
daily_residuals = (
    selected_predictions.groupby("order_date", as_index=False)
    .agg(residual=("residual", "sum"))
)

fig = px.bar(
    daily_residuals,
    x="order_date",
    y="residual",
    title=f"Daily Validation Residuals: {best_model_name}",
    labels={"order_date": "Date", "residual": "Actual - predicted"},
)
fig.add_hline(y=0, line_dash="dash")
fig.show()

In [20]:
market_residuals = (
    selected_predictions.groupby("market_name", as_index=False)
    .agg(
        actual=("actual", "sum"),
        predicted=("predicted", "sum"),
        residual=("residual", "sum"),
        abs_error=("abs_error", "sum"),
    )
)
market_residuals["wmape"] = market_residuals["abs_error"] / market_residuals["actual"]
market_residuals = market_residuals.sort_values("wmape", ascending=False)

category_residuals = (
    selected_predictions.groupby("category", as_index=False)
    .agg(
        actual=("actual", "sum"),
        predicted=("predicted", "sum"),
        residual=("residual", "sum"),
        abs_error=("abs_error", "sum"),
    )
)
category_residuals["wmape"] = category_residuals["abs_error"] / category_residuals["actual"]
category_residuals = category_residuals.sort_values("wmape", ascending=False)

display(market_residuals)
display(category_residuals)

,market_name,actual,predicted,residual,abs_error,wmape
10,New York,"672,192.0000","558,052.0023","114,139.9977","124,015.0080",0.1845
6,London,"641,349.0000","533,405.3530","107,943.6470","116,715.8544",0.1820
14,Sydney,"251,692.0000","214,350.7891","37,341.2109","45,764.2615",0.1818
15,Tokyo,"609,347.0000","533,848.5421","75,498.4579","103,552.8107",0.1699
5,Jakarta,"1,009,633.0000","957,464.7392","52,168.2608","171,031.5174",0.1694
7,Manila,"832,019.0000","770,482.6819","61,536.3181","140,161.0542",0.1685
0,Bangalore,"821,113.0000","719,977.5449","101,135.4551","133,367.1763",0.1624
13,Singapore,"714,102.0000","662,864.7935","51,237.2065","112,244.7201",0.1572
2,Dubai,"541,419.0000","470,397.7760","71,021.2240","82,716.3576",0.1528
9,Mumbai,"941,813.0000","829,098.6278","112,714.3722","139,810.3627",0.1484


,category,actual,predicted,residual,abs_error,wmape
1,E-Commerce,"2,878,623.0000","2,006,239.5931","872,383.4069","872,383.4069",0.3031
0,Digital Wallet,"1,384,869.0000","1,107,215.6526","277,653.3474","281,736.0870",0.2034
4,Ride Hailing,"2,220,066.0000","2,210,022.7558","10,043.2442","200,770.7744",0.0904
2,Food Delivery,"2,967,615.0000","2,962,170.3906","5,444.6094","235,921.9556",0.0795
3,Grocery,"1,398,827.0000","1,358,721.2575","40,105.7425","93,572.8702",0.0669


In [21]:
fig = px.bar(
    market_residuals.sort_values("wmape", ascending=True),
    x="wmape",
    y="market_name",
    orientation="h",
    title=f"Validation WMAPE by Market: {best_model_name}",
    labels={"wmape": "WMAPE", "market_name": "Market"},
)
fig.update_layout(xaxis_tickformat=".1%")
fig.show()

## 16. Feature Importance and Demand Predictors

Feature importance helps translate the model back into business language.

This section answers:

> Which features are most useful for predicting demand?

Interpret carefully:

- Importance shows predictive association, not causality.
- A feature can be important because it is directly useful, or because it acts as a proxy for market/category differences.
- Weather findings here complement the rain-lift chart; they do not replace it.

In [22]:
def get_feature_names(fitted_pipeline: Pipeline) -> np.ndarray:
    return fitted_pipeline.named_steps["preprocessor"].get_feature_names_out()


def feature_family(feature_name: str) -> str:
    base_name = feature_name.split("_")[0] if feature_name.startswith("category_") else feature_name

    if feature_name.startswith("market_id") or feature_name in ["nova_region", "country_iso3", "latitude", "longitude"]:
        return "Market identity"
    if feature_name.startswith("category") or feature_name == "category":
        return "Category"
    if feature_name in ["year", "quarter", "month", "day_of_week", "day_name", "is_weekend"] or feature_name.startswith("day_name"):
        return "Calendar"
    if feature_name in [
        "weather_code",
        "temperature_band",
        "precipitation_band",
        "temperature_2m_mean_c",
        "temperature_2m_max_c",
        "temperature_2m_min_c",
        "apparent_temperature_mean_c",
        "precipitation_sum_mm",
        "rain_sum_mm",
        "precipitation_hours",
        "wind_speed_10m_max_kmh",
        "is_rain_day",
    ] or feature_name.startswith(("weather_code", "temperature_band", "precipitation_band")):
        return "Weather"
    if feature_name in [
        "active_services",
        "avg_service_rating",
        "avg_popularity_score",
        "head_services",
        "mid_services",
        "long_tail_services",
        "head_service_share",
        "mid_service_share",
        "long_tail_service_share",
    ]:
        return "Service supply"
    if feature_name in [
        "market_weight",
        "growth_multiplier",
        "configured_ios_share",
        "population_total",
        "gdp_per_capita_current_usd",
        "urban_population_pct",
        "internet_users_pct",
        "mobile_subscriptions_per_100_people",
    ]:
        return "Market context"
    return "Other"


importance_tables = {}

if "random_forest_regression" in fitted_models:
    rf_pipeline = fitted_models["random_forest_regression"]
    rf_feature_names = get_feature_names(rf_pipeline)
    rf_importance = (
        pd.DataFrame(
            {
                "feature": rf_feature_names,
                "importance": rf_pipeline.named_steps["model"].feature_importances_,
            }
        )
        .assign(feature_family=lambda data: data["feature"].map(feature_family))
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    importance_tables["random_forest_regression"] = rf_importance
    display(rf_importance.head(25))

if "ridge_regression" in fitted_models:
    ridge_pipeline = fitted_models["ridge_regression"]
    ridge_feature_names = get_feature_names(ridge_pipeline)
    ridge_coef = (
        pd.DataFrame(
            {
                "feature": ridge_feature_names,
                "coefficient": ridge_pipeline.named_steps["model"].coef_,
            }
        )
        .assign(
            abs_coefficient=lambda data: data["coefficient"].abs(),
            feature_family=lambda data: data["feature"].map(feature_family),
        )
        .sort_values("abs_coefficient", ascending=False)
        .reset_index(drop=True)
    )
    importance_tables["ridge_regression"] = ridge_coef
    display(ridge_coef.head(25))

,feature,importance,feature_family
0,active_services,0.7621,Service supply
1,long_tail_services,0.0808,Service supply
2,mid_services,0.0503,Service supply
3,month,0.0403,Calendar
4,long_tail_service_share,0.0126,Service supply
5,is_weekend,0.0123,Calendar
6,quarter,0.0098,Calendar
7,avg_service_rating,0.0066,Service supply
8,head_service_share,0.0020,Service supply
9,category_Ride Hailing,0.0016,Category


,feature,coefficient,abs_coefficient,feature_family
0,long_tail_services,0.3025,0.3025,Service supply
1,mid_services,-0.2557,0.2557,Service supply
2,category_Digital Wallet,-0.2323,0.2323,Category
3,active_services,0.1968,0.1968,Service supply
4,category_Food Delivery,0.1466,0.1466,Category
5,category_Ride Hailing,0.1088,0.1088,Category
6,month,0.1013,0.1013,Calendar
7,category_Grocery,-0.0933,0.0933,Category
8,market_weight,0.0766,0.0766,Market context
9,category_E-Commerce,0.0701,0.0701,Category


In [23]:
if "random_forest_regression" in importance_tables:
    top_rf_importance = importance_tables["random_forest_regression"].head(20).sort_values("importance")

    fig = px.bar(
        top_rf_importance,
        x="importance",
        y="feature",
        orientation="h",
        title="Top Random Forest Feature Importances",
        labels={"importance": "Importance", "feature": "Feature"},
    )
    fig.show()
else:
    print("Random Forest feature importance is unavailable.")

### Validation-Based Permutation Importance

Built-in model importance can be helpful, but it does not directly answer how validation error changes when a feature is disrupted.

Permutation importance answers a more intuitive question:

> If we shuffle this feature in the validation data, how much worse does the model get?

Here we measure the increase in WMAPE after shuffling each original input feature. This is slower than built-in importance, but easier to explain in a presentation.

In [24]:
def predict_transactions(fitted_pipeline: Pipeline, X: pd.DataFrame) -> np.ndarray:
    return np.maximum(np.expm1(fitted_pipeline.predict(X)), 0)


def permutation_importance_wmape(
    fitted_pipeline: Pipeline,
    X: pd.DataFrame,
    y_true: pd.Series,
    feature_names: list[str],
    n_repeats: int = 3,
    random_state: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    baseline_pred = predict_transactions(fitted_pipeline, X)
    baseline_wmape = wmape(y_true, baseline_pred)
    rows = []

    for feature in feature_names:
        scores = []
        for _ in range(n_repeats):
            X_permuted = X.copy()
            X_permuted[feature] = rng.permutation(X_permuted[feature].to_numpy())
            permuted_pred = predict_transactions(fitted_pipeline, X_permuted)
            scores.append(wmape(y_true, permuted_pred))

        rows.append(
            {
                "feature": feature,
                "feature_family": feature_family(feature),
                "baseline_wmape": baseline_wmape,
                "permuted_wmape": float(np.mean(scores)),
                "wmape_increase": float(np.mean(scores) - baseline_wmape),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values("wmape_increase", ascending=False)
        .reset_index(drop=True)
    )


importance_model_name = best_model_name if best_model_name in fitted_models else "random_forest_regression"

if importance_model_name in fitted_models:
    permutation_importance_df = permutation_importance_wmape(
        fitted_pipeline=fitted_models[importance_model_name],
        X=X_valid,
        y_true=y_valid,
        feature_names=feature_cols,
        n_repeats=3,
        random_state=42,
    )
    print(f"Permutation importance model: {importance_model_name}")
    display(permutation_importance_df.head(20))
else:
    permutation_importance_df = pd.DataFrame()
    print("Permutation importance is unavailable because the selected model is the baseline.")

Permutation importance model: ridge_regression


,feature,feature_family,baseline_wmape,permuted_wmape,wmape_increase
0,mid_services,Service supply,0.1552,0.3824,0.2271
1,long_tail_services,Service supply,0.1552,0.3553,0.2000
2,active_services,Service supply,0.1552,0.2628,0.1075
3,category,Category,0.1552,0.2088,0.0535
4,market_weight,Market context,0.1552,0.1723,0.0171
5,latitude,Market identity,0.1552,0.1640,0.0087
6,is_weekend,Calendar,0.1552,0.1631,0.0078
7,apparent_temperature_mean_c,Weather,0.1552,0.1621,0.0069
8,growth_multiplier,Market context,0.1552,0.1606,0.0054
9,mid_service_share,Service supply,0.1552,0.1603,0.0050


### Recreating the Feature Family Importance Chart

This chart is built from `permutation_importance_df`, which measures how much validation WMAPE gets worse when each original feature is shuffled.

To recreate it after reopening the notebook, rerun these sections in order:

1. Setup and data load.
2. Feature selection.
3. Train and evaluate models.
4. Validation-based permutation importance.
5. Feature family importance.

The business-facing chart is the bar chart from `feature_family_importance`, not the built-in Random Forest feature importance chart. It is easier to explain because it groups individual predictors into themes like Service supply, Weather, Calendar, Market context, Category, and Market identity.

In [25]:
if not permutation_importance_df.empty:
    feature_family_importance = (
        permutation_importance_df.groupby("feature_family", as_index=False)
        .agg(
            total_wmape_increase=("wmape_increase", "sum"),
            avg_wmape_increase=("wmape_increase", "mean"),
            top_feature_wmape_increase=("wmape_increase", "max"),
            feature_count=("feature", "count"),
        )
        .sort_values("total_wmape_increase", ascending=False)
    )

    display(feature_family_importance)

    fig = px.bar(
        feature_family_importance.sort_values("total_wmape_increase", ascending=True),
        x="total_wmape_increase",
        y="feature_family",
        orientation="h",
        title="Feature Family Importance by Validation WMAPE Increase",
        labels={"total_wmape_increase": "Total WMAPE increase", "feature_family": "Feature family"},
    )
    fig.update_layout(xaxis_tickformat=".2%")
    fig.show()
else:
    feature_family_importance = pd.DataFrame()
    print("Feature family importance is unavailable.")

,feature_family,total_wmape_increase,avg_wmape_increase,top_feature_wmape_increase,feature_count
4,Service supply,0.5488,0.0610,0.2271,9
1,Category,0.0535,0.0535,0.0535,1
2,Market context,0.0347,0.0043,0.0171,8
3,Market identity,0.0167,0.0033,0.0087,5
5,Weather,0.0134,0.0011,0.0069,12
0,Calendar,0.0112,0.0019,0.0078,6


### Interpreting Service Supply as a Demand Predictor

If **Service supply** is the strongest feature family, the model is telling us that demand is highly associated with the depth and quality of available services in each market-category.

In this notebook, service supply includes:

- `active_services`: how many services are available in the market-category.
- `avg_service_rating`: the average rating of those services.
- `avg_popularity_score`: the average service popularity score.
- `head_services`, `mid_services`, `long_tail_services`: the number of services in each supply tier.
- `head_service_share`, `mid_service_share`, `long_tail_service_share`: the supply mix across tiers.

From a business perspective, this means daily demand is not explained only by external conditions like weather or calendar timing. Markets and categories with broader, stronger, and more mature service supply tend to have higher and more predictable demand.

This also fits the marketplace logic of a super-app: users are more likely to transact when there are enough relevant services available, when those services are highly rated, and when the category has a healthy mix of top-tier and long-tail providers.

One important caution: this is a **predictive association**, not proof of causality. Service supply may partly proxy for market maturity. Stronger markets can have both richer supply and stronger demand. To claim that adding more services causes demand growth, we would need a more causal analysis, such as an experiment or a before/after study around supply expansion.

## 17. Interpretation of Results

Use this section after running the notebook top-to-bottom.

The goal is to translate the model output into an analyst-friendly story:

1. Which model performed best?
2. Did it beat the simple baseline?
3. How large are the validation errors in transaction terms?
4. What are the strongest predictors of demand?
5. Which feature families matter most?
6. Which markets or categories have the highest residuals?
7. What should be investigated before using this in Tableau?

In [26]:
best_metrics = metrics_df.iloc[0]
baseline_metrics = metrics_df[metrics_df["model"] == "market_category_average_baseline"].iloc[0]

print("Best model summary")
print("------------------")
print(f"Best model: {best_metrics['model']}")
print(f"Validation MAE: {best_metrics['MAE']:,.2f} transactions")
print(f"Validation RMSE: {best_metrics['RMSE']:,.2f} transactions")
print(f"Validation WMAPE: {best_metrics['WMAPE']:.2%}")
print(f"Validation R2: {best_metrics['R2']:.3f}")
print()
print("Baseline comparison")
print("-------------------")
print(f"Baseline WMAPE: {baseline_metrics['WMAPE']:.2%}")
print(f"Best model WMAPE: {best_metrics['WMAPE']:.2%}")
print(f"WMAPE improvement vs baseline: {(baseline_metrics['WMAPE'] - best_metrics['WMAPE']):.2%}")

print()
print("Monthly dashboard output")
print("------------------------")
print(f"Monthly output rows: {len(monthly_dashboard_output):,}")
print(f"Months covered: {monthly_dashboard_output['year_month'].min()} to {monthly_dashboard_output['year_month'].max()}")
print(f"Monthly total WMAPE: {monthly_total['absolute_error'].sum() / monthly_total['actual_transactions'].sum():.2%}")

if not permutation_importance_df.empty:
    print()
    print("Top demand predictors by permutation importance")
    display(permutation_importance_df.head(10))

if not feature_family_importance.empty:
    print("Top feature families")
    display(feature_family_importance.head(10))

print()
print("Highest-error markets by validation WMAPE")
display(market_residuals.head(5))

print("Highest-error categories by validation WMAPE")
display(category_residuals.head(5))

Best model summary
------------------
Best model: ridge_regression
Validation MAE: 345.16 transactions
Validation RMSE: 520.58 transactions
Validation WMAPE: 15.52%
Validation R2: 0.727

Baseline comparison
-------------------
Baseline WMAPE: 27.84%
Best model WMAPE: 15.52%
WMAPE improvement vs baseline: 12.31%

Monthly dashboard output
------------------------
Monthly output rows: 160
Months covered: 2024-11 to 2024-12
Monthly total WMAPE: 15.52%

Top demand predictors by permutation importance


,feature,feature_family,baseline_wmape,permuted_wmape,wmape_increase
0,mid_services,Service supply,0.1552,0.3824,0.2271
1,long_tail_services,Service supply,0.1552,0.3553,0.2000
2,active_services,Service supply,0.1552,0.2628,0.1075
3,category,Category,0.1552,0.2088,0.0535
4,market_weight,Market context,0.1552,0.1723,0.0171
5,latitude,Market identity,0.1552,0.1640,0.0087
6,is_weekend,Calendar,0.1552,0.1631,0.0078
7,apparent_temperature_mean_c,Weather,0.1552,0.1621,0.0069
8,growth_multiplier,Market context,0.1552,0.1606,0.0054
9,mid_service_share,Service supply,0.1552,0.1603,0.0050


Top feature families


,feature_family,total_wmape_increase,avg_wmape_increase,top_feature_wmape_increase,feature_count
4,Service supply,0.5488,0.0610,0.2271,9
1,Category,0.0535,0.0535,0.0535,1
2,Market context,0.0347,0.0043,0.0171,8
3,Market identity,0.0167,0.0033,0.0087,5
5,Weather,0.0134,0.0011,0.0069,12
0,Calendar,0.0112,0.0019,0.0078,6



Highest-error markets by validation WMAPE


,market_name,actual,predicted,residual,abs_error,wmape
10,New York,"672,192.0000","558,052.0023","114,139.9977","124,015.0080",0.1845
6,London,"641,349.0000","533,405.3530","107,943.6470","116,715.8544",0.1820
14,Sydney,"251,692.0000","214,350.7891","37,341.2109","45,764.2615",0.1818
15,Tokyo,"609,347.0000","533,848.5421","75,498.4579","103,552.8107",0.1699
5,Jakarta,"1,009,633.0000","957,464.7392","52,168.2608","171,031.5174",0.1694


Highest-error categories by validation WMAPE


,category,actual,predicted,residual,abs_error,wmape
1,E-Commerce,"2,878,623.0000","2,006,239.5931","872,383.4069","872,383.4069",0.3031
0,Digital Wallet,"1,384,869.0000","1,107,215.6526","277,653.3474","281,736.0870",0.2034
4,Ride Hailing,"2,220,066.0000","2,210,022.7558","10,043.2442","200,770.7744",0.0904
2,Food Delivery,"2,967,615.0000","2,962,170.3906","5,444.6094","235,921.9556",0.0795
3,Grocery,"1,398,827.0000","1,358,721.2575","40,105.7425","93,572.8702",0.0669


## 18. Limitations and Next Steps

Important limitations:

- This is a first regression workflow, not a production forecasting system.
- The dataset is synthetic, so model findings should be treated as dashboard-supporting analysis rather than real-world causal proof.
- Feature importance shows model association, not causation.
- The validation period is only November and December 2024.
- We model daily demand levels, not demand growth.

Recommended next steps:

1. Review whether the best model meaningfully beats the baseline.
2. Use monthly actual vs predicted outputs for the dashboard narrative.
3. Use feature and feature-family importance to discuss demand predictors.
4. Inspect high-error markets and categories before using predictions in Tableau.
5. Decide which monthly prediction and residual outputs should become Tableau-ready tables.
6. Build clustering separately using a new market-grain dbt mart, instead of doing clustering feature aggregation inside this notebook.

## 19. Ingest Model Results into BigQuery

This final section writes the notebook outputs into the `dbt_doruk` dataset so they can be reused in Tableau.

The upload uses `pandas_gbq` and replaces the destination tables each time. Run it only after the notebook has been executed top-to-bottom and the outputs look correct.

In [27]:
MODEL_VERSION = "geo_demand_regression_v1"
DESTINATION_DATASET = DATASET_ID
GENERATED_AT_UTC = pd.Timestamp.utcnow()


def add_export_metadata(data: pd.DataFrame) -> pd.DataFrame:
    exported = data.copy()
    exported["model_version"] = MODEL_VERSION
    exported["generated_at_utc"] = GENERATED_AT_UTC
    return exported


metrics_export = add_export_metadata(
    metrics_df.drop(columns=[col for col in ["fitted_model"] if col in metrics_df.columns])
)

validation_predictions_export = add_export_metadata(selected_predictions.copy())
validation_predictions_export["order_date"] = pd.to_datetime(
    validation_predictions_export["order_date"]
).dt.date

monthly_dashboard_export = add_export_metadata(monthly_dashboard_output.copy())

feature_importance_export = add_export_metadata(permutation_importance_df.copy())

feature_family_importance_export = add_export_metadata(feature_family_importance.copy())

upload_tables = {
    "geo_demand_model_metrics": metrics_export,
    "geo_demand_model_validation_predictions": validation_predictions_export,
    "geo_demand_model_monthly_dashboard": monthly_dashboard_export,
    "geo_demand_model_feature_importance": feature_importance_export,
    "geo_demand_model_feature_family_importance": feature_family_importance_export,
}

for table_name, upload_df in upload_tables.items():
    if upload_df.empty:
        print(f"Skipping {table_name}: no rows to upload")
        continue

    destination_table = f"{DESTINATION_DATASET}.{table_name}"
    print(f"Uploading {len(upload_df):,} rows to {PROJECT_ID}.{destination_table}")
    pandas_gbq.to_gbq(
        dataframe=upload_df,
        destination_table=destination_table,
        project_id=PROJECT_ID,
        if_exists="replace",
        location=LOCATION,
    )

print("BigQuery ingest complete.")

Uploading 3 rows to nova-project-498911.dbt_doruk.geo_demand_model_metrics


/var/folders/28/8k26_pcd2bz699b6mx6vz2yc0000gn/T/ipykernel_41669/2712770783.py:3: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  GENERATED_AT_UTC = pd.Timestamp.utcnow()


Uploading 4,880 rows to nova-project-498911.dbt_doruk.geo_demand_model_validation_predictions
Uploading 160 rows to nova-project-498911.dbt_doruk.geo_demand_model_monthly_dashboard
Uploading 41 rows to nova-project-498911.dbt_doruk.geo_demand_model_feature_importance
Uploading 6 rows to nova-project-498911.dbt_doruk.geo_demand_model_feature_family_importance
BigQuery ingest complete.
